# Notebook 05 — Review Volume Trends

**Project:** Starbucks Customer Voice Intelligence: A U.S. Coffee Chain Market Study  
**Purpose:** Track how Starbucks review volume changed over 2017–2021 and whether that trajectory kept pace with the broader coffee market.

Review volume is a rough measure of customer engagement — not perfect, but useful. A sustained drop, even before ratings shift, usually means fewer people are bothering to react to a brand at all. The 2020 decline across all brands was COVID-19: dining restrictions, reduced foot traffic, fewer occasions to write a review.

## 0. Environment setup

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR  = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR  = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 1. Load data

Load `analysis_ready.parquet` and pull the columns needed for time-series aggregation.

In [2]:
df     = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
sbux   = df[df["brand_category"] == "Starbucks"]
market = df[df["brand_category"] != "Starbucks"]

print(f"Total reviews  : {len(df):,}")
print(f"Starbucks      : {len(sbux):,}")
print(f"Market         : {len(market):,}")
print(f"Date range     : {df['date'].min().date()} -> {df['date'].max().date()}")

Total reviews  : 381,999
Starbucks      : 11,675
Market         : 370,324
Date range     : 2017-01-01 -> 2021-12-31


## 2. Annual review volume

Total Starbucks reviews per year across the five-year window.

In [3]:
sbux_yr = sbux.groupby("year")["review_id"].count().reset_index()
sbux_yr.columns = ["year", "reviews"]

print("Starbucks annual review volume:")
print(sbux_yr.to_string(index=False))

Starbucks annual review volume:
 year  reviews
 2017     2536
 2018     2719
 2019     2996
 2020     1671
 2021     1753


In [4]:
fig = go.Figure(go.Bar(
    x=sbux_yr["year"].astype(str),
    y=sbux_yr["reviews"],
    text=sbux_yr["reviews"],
    textposition="outside",
    marker_color="#00704A",
    marker_line_color="#004d2e",
    marker_line_width=1,
))
fig.update_layout(
    title=dict(text="Starbucks — Annual Review Volume (2017–2021)", font=dict(size=16)),
    xaxis_title="Year",
    yaxis_title="Review Count",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", range=[0, 3500]),
    width=720, height=460,
    margin=dict(t=60, b=50, l=60, r=40)
)
fig.write_html(str(FIGURES_DIR / "05_volume_annual.html"))
fig.show()

## 3. Quarterly volume by year

Quarterly review counts for Starbucks, with the COVID-19 impact window (Q1–Q3 2020) visible in the 2020 line.

In [5]:
sbux_copy = sbux.copy()
sbux_copy["quarter_label"] = "Q" + sbux_copy["quarter"].astype(str)
vol = sbux_copy.groupby(["year", "quarter_label"])["review_id"].count().reset_index()
vol.columns = ["year", "quarter", "reviews"]

print("Starbucks quarterly volume:")
print(vol.to_string(index=False))

Starbucks quarterly volume:
 year quarter  reviews
 2017      Q1      650
 2017      Q2      662
 2017      Q3      606
 2017      Q4      618
 2018      Q1      648
 2018      Q2      672
 2018      Q3      707
 2018      Q4      692
 2019      Q1      681
 2019      Q2      779
 2019      Q3      774
 2019      Q4      762
 2020      Q1      555
 2020      Q2      222
 2020      Q3      435
 2020      Q4      459
 2021      Q1      399
 2021      Q2      432
 2021      Q3      449
 2021      Q4      473


In [6]:
YEAR_COLORS = {
    2017: "#b0d0e8",
    2018: "#6aaed6",
    2019: "#2171b5",
    2020: "#e63946",
    2021: "#f4a261",
}

fig = go.Figure()
for yr, grp in vol.groupby("year"):
    fig.add_trace(go.Scatter(
        x=grp["quarter"],
        y=grp["reviews"],
        mode="lines+markers",
        name=str(int(yr)),
        line=dict(color=YEAR_COLORS[int(yr)], width=2.5),
        marker=dict(size=8),
    ))

fig.update_layout(
    title=dict(text="Starbucks — Quarterly Review Volume by Year", font=dict(size=16)),
    xaxis_title="Quarter",
    yaxis_title="Review Count",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    legend_title="Year",
    width=820, height=480,
    margin=dict(t=60, b=50, l=60, r=40)
)
fig.write_html(str(FIGURES_DIR / "05_volume_by_year.html"))
fig.show()

## 4. Year-over-year growth comparison

Year-over-year volume change for Starbucks vs. the market. The question is whether Starbucks is keeping pace with the rest of the segment or losing ground.

In [7]:
sbux_yr_ct  = sbux.groupby("year")["review_id"].count()
mkt_yr_ct   = market.groupby("year")["review_id"].count()
sbux_yoy    = sbux_yr_ct.pct_change() * 100
mkt_yoy     = mkt_yr_ct.pct_change() * 100

yoy = pd.DataFrame({
    "Starbucks Reviews": sbux_yr_ct,
    "Starbucks YoY %":  sbux_yoy.round(1),
    "Market Reviews":   mkt_yr_ct,
    "Market YoY %":     mkt_yoy.round(1),
})
print(yoy.to_string())

      Starbucks Reviews  Starbucks YoY %  Market Reviews  Market YoY %
year                                                                  
2017               2536              NaN           81423           NaN
2018               2719              7.2           89393           9.8
2019               2996             10.2           87706          -1.9
2020               1671            -44.2           51847         -40.9
2021               1753              4.9           59955          15.6


## 5. Review volume by state

Geographic distribution of Starbucks reviews across the 13 states covered in this dataset.

> Note: State rankings reflect Yelp's user distribution, not Starbucks store density. California ranks 9th despite being Starbucks' largest physical market.

In [8]:
state_vol = sbux.groupby("state")["review_id"].count().reset_index()
state_vol.columns = ["state", "reviews"]
state_vol = state_vol.sort_values("reviews", ascending=False)

print("Review count by state:")
print(state_vol.to_string(index=False))

Review count by state:
state  reviews
   FL     2061
   PA     2042
   IN     1333
   NV     1155
   TN     1009
   MO      914
   AZ      906
   LA      753
   CA      483
   NJ      447
   DE      226
   ID      210
   IL      136


In [9]:
fig = px.choropleth(
    state_vol,
    locations="state",
    locationmode="USA-states",
    color="reviews",
    scope="usa",
    color_continuous_scale="YlOrRd",
    title="Starbucks — Review Volume by State (2017–2021)",
    labels={"reviews": "Reviews"}
)
fig.update_layout(
    width=900, height=520,
    paper_bgcolor="white",
    margin=dict(t=60, b=20, l=20, r=20),
    coloraxis_colorbar=dict(title="Reviews")
)
fig.write_html(str(FIGURES_DIR / "05_volume_by_state.html"))
fig.show()

In [10]:

# State-level average star rating for Starbucks
state_stars = sbux.groupby("state")["review_stars"].mean().reset_index()
state_stars.columns = ["state", "avg_stars"]
state_stars["avg_stars"] = state_stars["avg_stars"].round(2)

fig_s = px.choropleth(
    state_stars,
    locations="state",
    locationmode="USA-states",
    color="avg_stars",
    scope="usa",
    color_continuous_scale="RdYlGn",
    range_color=[2.0, 4.0],
    title="Starbucks — Avg Star Rating by State (2017–2021)",
    labels={"avg_stars": "Avg ★"}
)
fig_s.update_layout(
    width=900, height=520,
    paper_bgcolor="white",
    margin=dict(t=60, b=20, l=20, r=20),
    coloraxis_colorbar=dict(title="Avg ★"),
)
fig_s.write_html(str(FIGURES_DIR / "05_rating_by_state.html"))
fig_s.show()

print("\nAvg star rating by state:")
print(state_stars.sort_values("avg_stars", ascending=False).to_string(index=False))



Avg star rating by state:
state  avg_stars
   IN       3.28
   ID       3.15
   FL       3.06
   AZ       3.04
   IL       2.92
   MO       2.86
   NJ       2.86
   NV       2.80
   CA       2.79
   PA       2.79
   LA       2.67
   TN       2.59
   DE       2.55


## Key findings

- 2019 was the high point: Q2 hit 779 reviews, up 10.2% while the broader market shrank 1.9%. Starbucks was one of the few brands in the segment gaining ground that year.
- Q2 2020 fell to 222 reviews (-44.2%). The market dropped 40.9% in the same quarter. Both numbers trace to COVID-19 — dining restrictions, closed stores, fewer occasions to write a review.
- The recovery gap is worth watching. The market bounced back 15.6% in 2021; Starbucks managed 4.9%, and volume had not returned to 2019 levels by the end of the dataset window.
- Only 13 states appear in this dataset. FL and PA lead in review count; CA ranks 9th. That ranking reflects Yelp's geographic user base, not store density or performance.

---

**Next: Notebook 06 — Rating Distribution**